In [ ]:
import json
import polars as pl
import torch
from open_clip import create_model_from_pretrained, get_tokenizer
from torch.nn.attention import SDPBackend, sdpa_kernel
from torch.utils.data import DataLoader
from tqdm import tqdm

import sys

sys.path.append("..")

from src.datasets.mp16 import ImageDataset
from src.datasets.classify import DATASET_PATH_BUILDERS, batch_agg_score

In [11]:
MODEL_NAME = "hf-hub:timm/ViT-gopt-16-SigLIP2-384"
device = "cuda"

In [ ]:
def build_text_embeddings(
    model, tokenizer, desc_flatten: list[str], device: str
) -> torch.Tensor:
    """Pre-compute normalized text embeddings for all label descriptions."""
    text = tokenizer(desc_flatten, context_length=model.context_length).to(device)

    with (
        torch.no_grad(),
        torch.amp.autocast(device),
        sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION),
    ):
        text_embs = model.encode_text(text, normalize=True)

    return text_embs  # [N_desc, D]

In [ ]:
# Load model + preprocessor
model, preprocess = create_model_from_pretrained(MODEL_NAME)
tokenizer = get_tokenizer(MODEL_NAME)

model = model.to(device).eval()
model = torch.compile(model)

# Load labels
labels = json.loads(open("labels_simplified.json").read())
categories = [
    f"This is a photo of {label.replace('_', ' ')}." for label in labels.keys()
]
desc_flatten = [f"This is a photo of {desc}." for desc in labels.values()]
desc_to_cat = {d: c for c, d in labels.items()}

# Pre-compute text embeddings once
print("Computing text embeddings...")
text_embs = build_text_embeddings(model, tokenizer, categories, device)


open_clip_config.json:   0%|          | 0.00/984 [00:00<?, ?B/s]

open_clip_model.safetensors:   0%|          | 0.00/7.49G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

Computing text embeddings...


In [ ]:
df = pl.read_csv("../../datasets/google-landmark/raw-csv/index_image_to_landmark.csv")
img_ids = df["id"].to_list()
img_paths = DATASET_PATH_BUILDERS["gld"](
    df, "id", "/mnt/yokoyamalab-nas/gldv2-full/index"
)

dataset = ImageDataset(preprocess, img_paths=img_paths, img_ids=img_ids)
loader = DataLoader(
    dataset,
    batch_size=64,
    num_workers=4,
    pin_memory=True,
    drop_last=False,
    persistent_workers=True,
    prefetch_factor=4 // 2,
)

In [ ]:
all_outputs: list[list[dict]] = []
all_img_ids: list[str] = []

with (
    torch.no_grad(),
    torch.amp.autocast(device),
    sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION),
):
    for images, ids in tqdm(loader, desc="classify"):
        img_embs = model.encode_image(images.to(device), normalize=True)
        scores = torch.sigmoid(
            img_embs @ text_embs.T * model.logit_scale.exp() + model.logit_bias
        )

        for row in scores.cpu().float().tolist():
            all_outputs.append(
                [
                    {"label": desc, "score": score}
                    for desc, score in zip(labels.keys(), row)
                ]
            )
        all_img_ids.extend(ids)

        break

classify:   0%|          | 0/11903 [00:01<?, ?it/s]


In [16]:
sorted(all_outputs[0], key=lambda x: x["score"], reverse=True)

[{'label': 'monument_statue', 'score': 0.0008492469787597656},
 {'label': 'church_cathedral', 'score': 0.0005154609680175781},
 {'label': 'castle_fortress', 'score': 6.759166717529297e-05},
 {'label': 'ruins', 'score': 2.7120113372802734e-05},
 {'label': 'tower', 'score': 2.4318695068359375e-05},
 {'label': 'temple_shrine', 'score': 1.2934207916259766e-05},
 {'label': 'theater_opera', 'score': 1.0132789611816406e-05},
 {'label': 'fountain', 'score': 9.5367431640625e-06},
 {'label': 'museum_gallery', 'score': 8.821487426757812e-06},
 {'label': 'bridge', 'score': 8.58306884765625e-06},
 {'label': 'palace_manor', 'score': 3.874301910400391e-06},
 {'label': 'mosque', 'score': 3.2782554626464844e-06},
 {'label': 'arch_gate', 'score': 1.0728836059570312e-06},
 {'label': 'modern_landmark', 'score': 7.152557373046875e-07},
 {'label': 'stadium_arena', 'score': 2.384185791015625e-07},
 {'label': 'harbor_pier', 'score': 1.1920928955078125e-07},
 {'label': 'railway_station', 'score': 5.96046447753

In [ ]:
import timm

model = timm.create_model(
    "timm/convnext_xxlarge.clip_laion2b_soup_ft_in12k", pretrained=True
)
model = model.eval()

# get model specific transforms (normalization, resize)
data_config = timm.data.resolve_model_data_config(model)
transforms = timm.data.create_transform(**data_config, is_training=False)

# output = model(transforms(img).unsqueeze(0))  # unsqueeze single image into batch of 1

# top5_probabilities, top5_class_indices = torch.topk(output.softmax(dim=1) * 100, k=5)

: 

In [ ]:
import timm.data                                                   
                
info = timm.data.ImageNetInfo("imagenet-12k")  # default: imagenet-1k       
                
# Get all label descriptions as a list                        
descriptions = info.la bel_descriptions()
                                                            
# As a dict {label_name: description}                         
descriptions_dict = info.label_descriptions(as_dict=True)
                                                            
# Detailed descriptions                                       
detailed = info.label_descriptions(detailed=True)

In [70]:
descriptions_dict["n03767745"]

'minaret'

In [64]:
with open("./labels.txt", "w") as f:
    for desc in descriptions:
        f.write(desc + "\n")

In [ ]:
from PIL import Image

with torch.no_grad():
    out = model(transforms(Image.open(dataset.img_paths[0])).unsqueeze(0))

In [77]:
pred = torch.topk(out.softmax(dim=1) * 100, k=10).indices.view(-1).tolist()


In [80]:
pred

[11771, 6277, 4102, 6276, 4084, 6737, 5921, 4783, 2889, 3305]

In [82]:
descriptions[pred[1]]

'shield'

In [79]:
for p in pred:
    print(info.index_to_description(p))

stemma
shield
escutcheon, scutcheon
shield, buckler
ensign
sword, blade, brand, steel
rapier, tuck
imprint
bell
cartouche, cartouch


In [ ]:
import polars as pl
from PIL import Image

df = pl.read_csv("../../datasets/google-landmark/raw-csv/index_set_fixed.csv")
df

id,landmark_id,latitude,longitude,wikimedia_url,geohack_url
str,i64,f64,f64,str,str
"""fdf40612109ad174""",32888,33.486266,-80.860517,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…"
"""5a6cc67c893daea6""",552,48.851075,2.285447,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…"
"""87b88acb68cdc1f1""",13626,40.758972,-73.979389,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…"
"""c4ac217ce087b251""",8699,42.097,-72.5633,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…"
"""05f269bf32be9d3e""",30838,50.681278,21.749444,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…"
…,…,…,…,…,…
"""ace1f52eea7620e9""",1463,53.001714,6.72987,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…"
"""f07d97acbc9cac79""",51420,52.377329,4.59569,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…"
"""2a2bdc07e5144f71""",27922,17.242778,98.972222,"""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…"


In [17]:
with open("../../index-img.txt", "w") as f:
    for item in df.select("id").to_numpy().squeeze().tolist():
        f.write(f"{item}.jpg\n")

In [25]:
train = pl.read_csv("../../datasets/google-landmark/raw-csv/train_label_to_category.csv")
train.sample(25).write_csv("../sample.csv")

In [26]:
train

landmark_id,category
i64,str
0,"""http://commons.wikimedia.org/w…"
1,"""http://commons.wikimedia.org/w…"
2,"""http://commons.wikimedia.org/w…"
3,"""http://commons.wikimedia.org/w…"
4,"""http://commons.wikimedia.org/w…"
…,…
203089,"""http://commons.wikimedia.org/w…"
203090,"""http://commons.wikimedia.org/w…"
203091,"""http://commons.wikimedia.org/w…"
